# 05 - Interactive Dashboard
## AI-Based Pharmaceutical Data Selection and Monitoring System
**Capstone Project PJK-GM016 | Pijak x IBM SkillsBuild**

---

### Struktur Dashboard (Streamlit Web App)

| Tab | Konten |
|-----|--------|
| [1] Overview | KPI cards, distribusi defect per bulan/line, parameter kritis, tabel data |
| [2] Monitoring | Tren yield per batch, moving average, anomaly score, scatter suhu vs yield |
| [3] Analisis Defect | Frekuensi penyebab, detail per batch dengan reason cards, heatmap korelasi |
| [4] Model AI | Re-train RF/GB/IF, metrik evaluasi, confusion matrix, feature importance |
| [5] Clustering | Elbow method, PCA 2D scatter, profil cluster |

> **Fitur Utama:**
> - Upload dataset (CSV) langsung dari browser tanpa perlu model file
> - Filter dinamis per bulan, line CB, dan status batch
> - Re-train model ML on-the-fly dari data yang diupload
> - Visualisasi penyebab defect per batch dengan reason cards
> - KPI cards, charts interaktif, confusion matrix, feature importance


## 0. Install Dependensi


In [1]:
import subprocess, sys

required = ['streamlit', 'plotly', 'scikit-learn', 'pandas', 'numpy', 'openpyxl']
for pkg in required:
    try:
        __import__(pkg.replace('-','_'))
        print(f'  OK  {pkg}')
    except ImportError:
        print(f'  Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
        print(f'  Installed {pkg}')

print('\nSemua dependensi siap.')


  OK  streamlit
  OK  plotly
  Installing scikit-learn...
  Installed scikit-learn
  OK  pandas
  OK  numpy
  OK  openpyxl

Semua dependensi siap.


## 1. Cek Ketersediaan File


In [2]:
import os, pandas as pd

print('=' * 55)
print('  CEK FILE DATASET')
print('=' * 55)

data_files = ['dataset_with_predictions.csv', 'dataset_clean.csv']

print('\nDataset Files:')
found_any = False
for f in data_files:
    if os.path.exists(f):
        df_c = pd.read_csv(f, low_memory=False)
        print(f'  FOUND  {f}  ->  {df_c.shape[0]} baris x {df_c.shape[1]} kolom')
        found_any = True
    else:
        print(f'  NOT FOUND  {f}')

if not found_any:
    print('\n  INFO: Upload CSV via sidebar saat dashboard berjalan.')

print('\nCek selesai.')


  CEK FILE DATASET

Dataset Files:
  FOUND  dataset_with_predictions.csv  ->  46 baris x 150 kolom
  FOUND  dataset_clean.csv  ->  46 baris x 135 kolom

Cek selesai.


## 2. Generate `app.py`

Cell ini menulis `app.py` ke disk.  
Setelah selesai, buka terminal dan jalankan:
```bash
streamlit run app.py
```


In [3]:
import os

# Isi app.py disimpan sebagai list of lines
APP_LINES = [
    "import streamlit as st",
    "import pandas as pd",
    "import numpy as np",
    "import plotly.express as px",
    "import plotly.graph_objects as go",
    "from plotly.subplots import make_subplots",
    "import warnings",
    "import re",
    "import io",
    "from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, IsolationForest",
    "from sklearn.preprocessing import StandardScaler, LabelEncoder",
    "from sklearn.decomposition import PCA",
    "from sklearn.metrics import (accuracy_score, precision_score, recall_score,",
    "                             f1_score, confusion_matrix)",
    "",
    "warnings.filterwarnings(\"ignore\")",
    "",
    "# -- Page Config ----------------------------------------------------------------",
    "st.set_page_config(",
    "    page_title=\"AI Pharma Monitoring | PJK-GM016\",",
    "    page_icon=\"[APP]\",",
    "    layout=\"wide\",",
    "    initial_sidebar_state=\"expanded\"",
    ")",
    "",
    "# -- Custom CSS -----------------------------------------------------------------",
    "st.markdown(\"\"\"",
    "<style>",
    "  @import url('https://fonts.googleapis.com/css2?family=Syne:wght@400;600;700;800&family=DM+Sans:wght@300;400;500&display=swap');",
    "",
    "  html, body, [class*=\"css\"] { font-family: 'DM Sans', sans-serif; }",
    "  h1, h2, h3, .big-title { font-family: 'Syne', sans-serif; }",
    "",
    "  /* -- Sidebar -- */",
    "  [data-testid=\"stSidebar\"] {",
    "    background: linear-gradient(180deg, #0f172a 0%, #1e293b 100%);",
    "    border-right: 1px solid #334155;",
    "  }",
    "  [data-testid=\"stSidebar\"] * { color: #e2e8f0 !important; }",
    "  [data-testid=\"stSidebar\"] .stSelectbox label,",
    "  [data-testid=\"stSidebar\"] .stMultiSelect label { color: #94a3b8 !important; font-size:0.78rem; }",
    "",
    "  /* -- Main bg -- */",
    "  .stApp { background: #f8fafc; }",
    "",
    "  /* -- KPI Cards -- */",
    "  .kpi-wrap { display:flex; flex-wrap:wrap; gap:14px; margin-bottom:20px; }",
    "  .kpi-card {",
    "    flex:1 1 150px; border-radius:16px; padding:20px 18px;",
    "    color:white; position:relative; overflow:hidden;",
    "    box-shadow: 0 4px 20px rgba(0,0,0,0.12);",
    "  }",
    "  .kpi-card::after {",
    "    content:''; position:absolute; right:-20px; bottom:-20px;",
    "    width:80px; height:80px; border-radius:50%;",
    "    background:rgba(255,255,255,0.08);",
    "  }",
    "  .kpi-label { font-size:0.72rem; text-transform:uppercase; letter-spacing:1.5px; opacity:0.82; }",
    "  .kpi-value { font-family:'Syne',sans-serif; font-size:2.1rem; font-weight:800; margin:4px 0 2px; }",
    "  .kpi-sub   { font-size:0.75rem; opacity:0.75; }",
    "  .kpi-blue  { background: linear-gradient(135deg,#3b82f6,#1d4ed8); }",
    "  .kpi-green { background: linear-gradient(135deg,#10b981,#065f46); }",
    "  .kpi-red   { background: linear-gradient(135deg,#ef4444,#991b1b); }",
    "  .kpi-amber { background: linear-gradient(135deg,#f59e0b,#b45309); }",
    "  .kpi-purple{ background: linear-gradient(135deg,#8b5cf6,#4c1d95); }",
    "  .kpi-teal  { background: linear-gradient(135deg,#14b8a6,#0f766e); }",
    "",
    "  /* -- Section heading -- */",
    "  .section-title {",
    "    font-family:'Syne',sans-serif; font-size:1.1rem; font-weight:700;",
    "    color:#1e293b; border-left:4px solid #3b82f6; padding-left:12px;",
    "    margin:28px 0 14px;",
    "  }",
    "",
    "  /* -- Status pills -- */",
    "  .pill-normal {",
    "    background:#dcfce7; color:#166534; padding:3px 12px;",
    "    border-radius:20px; font-size:0.78rem; font-weight:600;",
    "  }",
    "  .pill-defect {",
    "    background:#fee2e2; color:#991b1b; padding:3px 12px;",
    "    border-radius:20px; font-size:0.78rem; font-weight:600;",
    "  }",
    "",
    "  /* -- Upload box -- */",
    "  .upload-hero {",
    "    border:2px dashed #3b82f6; border-radius:16px;",
    "    padding:40px 30px; text-align:center;",
    "    background: linear-gradient(135deg,#eff6ff,#f0fdf4);",
    "  }",
    "",
    "  /* -- Reason card -- */",
    "  .reason-card {",
    "    background:#fff; border:1px solid #e2e8f0; border-radius:12px;",
    "    padding:14px 18px; margin-bottom:10px;",
    "    border-left:4px solid #ef4444;",
    "    box-shadow:0 2px 8px rgba(0,0,0,0.04);",
    "  }",
    "  .reason-badge {",
    "    display:inline-block; background:#fee2e2; color:#991b1b;",
    "    font-size:0.72rem; font-weight:600; padding:2px 8px;",
    "    border-radius:10px; margin-right:6px;",
    "  }",
    "  .reason-text { color:#374151; font-size:0.88rem; margin-top:4px; }",
    "",
    "  /* -- Tab styling -- */",
    "  [data-testid=\"stTabs\"] [data-baseweb=\"tab-list\"] {",
    "    background: #f1f5f9; border-radius:12px; padding:4px; gap:4px;",
    "  }",
    "  [data-testid=\"stTabs\"] [data-baseweb=\"tab\"] {",
    "    border-radius:8px; font-family:'Syne',sans-serif;",
    "    font-weight:600; font-size:0.82rem;",
    "  }",
    "",
    "  /* -- Footer -- */",
    "  .footer {",
    "    text-align:center; color:#94a3b8; font-size:0.75rem;",
    "    margin-top:40px; padding-top:20px; border-top:1px solid #e2e8f0;",
    "  }",
    "</style>",
    "\"\"\", unsafe_allow_html=True)",
    "",
    "# -- Constants ------------------------------------------------------------------",
    "C_NORMAL  = \"#10b981\"",
    "C_DEFECT  = \"#ef4444\"",
    "C_BLUE    = \"#3b82f6\"",
    "C_AMBER   = \"#f59e0b\"",
    "C_PURPLE  = \"#8b5cf6\"",
    "BG_DARK   = \"#0f172a\"",
    "BG_PLOT   = \"#1e293b\"",
    "GRID      = \"#334155\"",
    "",
    "PLOT_LAYOUT = dict(",
    "    paper_bgcolor=\"rgba(0,0,0,0)\",",
    "    plot_bgcolor =\"#f8fafc\",",
    "    font=dict(color=\"#374151\", family=\"DM Sans\"),",
    "    margin=dict(t=40, b=30, l=20, r=20),",
    "    xaxis=dict(gridcolor=\"#e2e8f0\", linecolor=\"#e2e8f0\"),",
    "    yaxis=dict(gridcolor=\"#e2e8f0\", linecolor=\"#e2e8f0\"),",
    "    legend=dict(bgcolor=\"rgba(255,255,255,0.9)\", bordercolor=\"#e2e8f0\",",
    "                borderwidth=1, font=dict(size=11)),",
    ")",
    "",
    "# -- Session State --------------------------------------------------------------",
    "if \"df\" not in st.session_state:",
    "    st.session_state.df = None",
    "if \"df_source\" not in st.session_state:",
    "    st.session_state.df_source = None   # \"predictions\" | \"clean\" | \"raw\"",
    "",
    "# ========================================================================",
    "# HELPERS",
    "# ========================================================================",
    "",
    "def detect_and_prepare(df_raw: pd.DataFrame):",
    "    \"\"\"",
    "    Deteksi format dataset (predictions / clean) dan siapkan kolom standar.",
    "    Returns (df, source_type)",
    "    \"\"\"",
    "    df = df_raw.copy()",
    "",
    "    # -- Format A: dataset_with_predictions.csv -------------------------",
    "    if \"is_defect\" in df.columns and \"defect_reasons\" in df.columns:",
    "        df[\"label_display\"] = df[\"is_defect\"].map({0:\"Normal\",1:\"Defect\"}).fillna(\"Normal\")",
    "        # model consensus",
    "        if \"model_consensus_pred\" in df.columns:",
    "            df[\"pred_label\"] = df[\"model_consensus_pred\"].map({0:\"Normal\",1:\"Defect\"}).fillna(df[\"label_display\"])",
    "        else:",
    "            df[\"pred_label\"] = df[\"label_display\"]",
    "        # row_no fallback",
    "        if \"row_no\" not in df.columns:",
    "            df[\"row_no\"] = range(1, len(df)+1)",
    "        return df, \"predictions\"",
    "",
    "    # -- Format B: dataset_clean.csv ------------------------------------",
    "    if \"cb1_suhu_rata\" in df.columns or \"cb1_pct_yield\" in df.columns:",
    "        # Hitung is_defect jika belum ada",
    "        if \"is_defect\" not in df.columns:",
    "            conds = []",
    "            if \"cb1_suhu_rata\" in df.columns:",
    "                conds.append(df[\"cb1_suhu_rata\"] > 75)",
    "            if \"cb2_suhu_rata\" in df.columns:",
    "                conds.append(df[\"cb2_suhu_rata\"] > 75)",
    "            if \"cb1_pct_yield\" in df.columns:",
    "                conds.append(df[\"cb1_pct_yield\"] < 99)",
    "            if \"ck1_pct_yield\" in df.columns:",
    "                conds.append(df[\"ck1_pct_yield\"] < 99)",
    "            if conds:",
    "                df[\"is_defect\"] = np.where(np.logical_or.reduce(conds), 1, 0)",
    "            else:",
    "                df[\"is_defect\"] = 0",
    "        df[\"label_display\"] = df[\"is_defect\"].map({0:\"Normal\",1:\"Defect\"}).fillna(\"Normal\")",
    "        df[\"pred_label\"]    = df[\"label_display\"]",
    "        if \"defect_reasons\" not in df.columns:",
    "            df[\"defect_reasons\"] = \"-\"",
    "        if \"row_no\" not in df.columns:",
    "            df[\"row_no\"] = range(1, len(df)+1)",
    "        return df, \"clean\"",
    "",
    "    return df, \"raw\"",
    "",
    "", 
    "def run_ml_pipeline(df: pd.DataFrame):",
    "    \"\"\"Re-train RF + Isolation Forest dan kembalikan prediksi.\"\"\"",
    "    exclude_kw = [\"pred\",\"proba\",\"probability\",\"status\",\"label\",\"consensus\",",
    "                  \"defect_reasons\",\"reasons\",\"enc\",\"num\",\"row_no\"]",
    "    exclude_cols = [\"is_defect\",\"label_display\",\"pred_label\",",
    "                    \"material_desc\",\"cb1_bulan\",\"cb2_bulan\",",
    "                    \"cb1_line\",\"cb2_line\",\"cb1_shift\",\"cb2_shift\",",
    "                    \"ck1_keterangan\",\"ck2_keterangan\",",
    "                    \"lapis1_warna\",\"lapis2_warna\"]",
    "",
    "    feat_cols = [c for c in df.columns",
    "                 if c not in exclude_cols",
    "                 and not any(k in c.lower() for k in exclude_kw)",
    "                 and df[c].dtype in [np.float64, np.int64, float, int]]",
    "",
    "    if not feat_cols or \"is_defect\" not in df.columns:",
    "        return df, None, feat_cols",
    "",
    "    df_m = df[feat_cols + [\"is_defect\"]].copy()",
    "    df_m = df_m.apply(lambda s: s.fillna(s.median()), axis=0)",
    "",
    "    X = df_m[feat_cols].values",
    "    y = df_m[\"is_defect\"].astype(int).values",
    "",
    "    scaler = StandardScaler()",
    "    Xs = scaler.fit_transform(X)",
    "",
    "    # Random Forest",
    "    rf = RandomForestClassifier(n_estimators=200, max_depth=8,",
    "                                 random_state=42, class_weight=\"balanced\")",
    "    rf.fit(Xs, y)",
    "    df[\"rf_pred\"]        = rf.predict(Xs)",
    "    df[\"rf_proba_defect\"] = rf.predict_proba(Xs)[:, 1]",
    "    df[\"rf_status\"]      = df[\"rf_pred\"].map({0:\"Normal\",1:\"Defect\"})",
    "",
    "    # Isolation Forest",
    "    iso = IsolationForest(contamination=0.15, random_state=42, n_estimators=200)",
    "    iso.fit(Xs)",
    "    iso_pred = iso.predict(Xs)",
    "    df[\"isolation_forest_score\"] = iso.score_samples(Xs)",
    "    df[\"if_pred\"]  = np.where(iso_pred == -1, 1, 0)",
    "    df[\"if_status\"] = df[\"if_pred\"].map({0:\"Normal\",1:\"Defect\"})",
    "",
    "    # Gradient Boosting",
    "    gb = GradientBoostingClassifier(n_estimators=100, max_depth=4,",
    "                                     random_state=42)",
    "    gb.fit(Xs, y)",
    "    df[\"gb_pred\"]        = gb.predict(Xs)",
    "    df[\"gb_proba_defect\"] = gb.predict_proba(Xs)[:, 1]",
    "    df[\"gb_status\"]      = df[\"gb_pred\"].map({0:\"Normal\",1:\"Defect\"})",
    "",
    "    # Consensus",
    "    df[\"model_vote_defect\"]  = df[\"rf_pred\"] + df[\"if_pred\"] + df[\"gb_pred\"]",
    "    df[\"model_consensus_pred\"] = (df[\"model_vote_defect\"] >= 2).astype(int)",
    "    df[\"pred_label\"] = df[\"model_consensus_pred\"].map({0:\"Normal\",1:\"Defect\"})",
    "",
    "    fi = pd.DataFrame({\"feature\":feat_cols, \"importance\":rf.feature_importances_})",
    "    fi = fi.sort_values(\"importance\", ascending=False).reset_index(drop=True)",
    "",
    "    return df, fi, feat_cols",
    "",
    "",
    "def kpi_html(label, value, sub, cls):",
    "    return f\"\"\"<div class='kpi-card {cls}'>",
    "  <div class='kpi-label'>{label}</div>",
    "  <div class='kpi-value'>{value}</div>",
    "  <div class='kpi-sub'>{sub}</div>",
    "</div>\"\"\"",
    "",
    "",
    "def render_kpis(df):",
    "    total   = len(df)",
    "    n_def   = int(df[\"is_defect\"].sum())",
    "    n_norm  = total - n_def",
    "    dr      = n_def / total * 100 if total else 0",
    "    avg_y1  = df[\"cb1_pct_yield\"].mean()  if \"cb1_pct_yield\"  in df.columns else None",
    "    avg_y2  = df[\"ck1_pct_yield\"].mean()  if \"ck1_pct_yield\"  in df.columns else None",
    "    avg_s1  = df[\"cb1_suhu_rata\"].mean()  if \"cb1_suhu_rata\"  in df.columns else None",
    "    avg_ka  = df[\"cb1_rata_ka\"].mean()    if \"cb1_rata_ka\"    in df.columns else None",
    "",
    "    cards = [",
    "        kpi_html(\"Total Batch\",   str(total),                  \"batch dianalisis\",          \"kpi-blue\"),",
    "        kpi_html(\"Batch Normal\",  str(n_norm),                 f\"{100-dr:.1f}% dari total\", \"kpi-green\"),",
    "        kpi_html(\"Batch Defect\",  str(n_def),                  f\"{dr:.1f}% defect rate\",    \"kpi-red\"),",
    "    ]",
    "    if avg_y1 is not None:",
    "        cards.append(kpi_html(\"Avg Yield CB1\", f\"{avg_y1:.1f}%\", \"target \u2265 99%\",     \"kpi-amber\"))",
    "    if avg_y2 is not None:",
    "        cards.append(kpi_html(\"Avg Yield CK1\", f\"{avg_y2:.1f}%\", \"setelah pengeringan\", \"kpi-purple\"))",
    "    if avg_s1 is not None:",
    "        cards.append(kpi_html(\"Avg Suhu CB1\",  f\"{avg_s1:.1f}\u00b0C\",\"batas \u2264 75\u00b0C\",     \"kpi-teal\"))",
    "",
    "    st.markdown(\"<div class='kpi-wrap'>\" + \"\".join(cards) + \"</div>\",",
    "                unsafe_allow_html=True)",
    "",
    "",
    "def parse_reasons(series):",
    "    \"\"\"Kembalikan Series frekuensi alasan dari kolom defect_reasons.\"\"\"",
    "    reasons = []",
    "    for val in series.dropna():",
    "        for r in str(val).split(\"|\"):",
    "            r = re.sub(r\"\\(.*?\\)\", \"\", r).strip()",
    "            if r and r not in (\"-\", \"nan\", \"\"):",
    "                reasons.append(r)",
    "    return pd.Series(reasons).value_counts() if reasons else pd.Series(dtype=int)",
    "",
    "",
    "# ========================================================================",
    "# SIDEBAR",
    "# ========================================================================",
    "with st.sidebar:",
    "    st.markdown(\"\"\"",
    "    <div style='text-align:center; padding:10px 0 20px'>",
    "      <div style='font-family:Syne,sans-serif; font-size:1.4rem; font-weight:800;",
    "                  background:linear-gradient(135deg,#3b82f6,#8b5cf6);",
    "                  -webkit-background-clip:text; -webkit-text-fill-color:transparent;'>",
    "        [APP] AI Pharma",
    "      </div>",
    "      <div style='font-size:0.72rem; color:#64748b; margin-top:2px;'>",
    "        Monitoring System * PJK-GM016",
    "      </div>",
    "    </div>",
    "    \"\"\", unsafe_allow_html=True)",
    "",
    "    st.markdown(\"### [DIR] Upload Dataset\")",
    "    uploaded = st.file_uploader(",
    "        \"CSV / Excel dataset granulasi\",",
    "        type=[\"csv\",\"xlsx\",\"xls\"],",
    "        help=\"Upload file raw Excel (.xlsx/.xls) atau CSV hasil preprocessing\"",
    "    )",
    "",
    "    if uploaded:",
    "        try:",
    "            fname = uploaded.name.lower()",
    "            if fname.endswith((\".xlsx\",\".xls\")):",
    "                df_up = pd.read_excel(uploaded)",
    "                st.info(f\"📂 File Excel: {uploaded.name} — {len(df_up)} baris\")",
    "            else:",
    "                df_up = pd.read_csv(uploaded, low_memory=False)",
    "            df_up, src = detect_and_prepare(df_up)",
    "            if src == \"raw\":",
    "                st.warning(\"⚠️ Format tidak dikenali. Pastikan kolom granulasi tersedia.\")",
    "            st.session_state.df = df_up",
    "            st.session_state.df_source = src",
    "            st.success(f\"[OK] {len(df_up)} baris dimuat\")",
    "            st.caption(f\"Format terdeteksi: **{src}**\")",
    "        except Exception as e:",
    "            st.error(f\"Gagal memuat file: {e}\")",
    "",
    "    st.divider()",
    "",
    "    df = st.session_state.df",
    "",
    "    if df is not None:",
    "        st.markdown(\"### [FILTER] Filter Data\")",
    "        if \"cb1_bulan\" in df.columns:",
    "            bulan_opts = [\"Semua\"] + sorted(df[\"cb1_bulan\"].dropna().unique().tolist())",
    "            sel_bulan  = st.multiselect(\"Bulan\", bulan_opts, default=[\"Semua\"])",
    "        else:",
    "            sel_bulan = [\"Semua\"]",
    "",
    "        if \"cb1_line\" in df.columns:",
    "            line_opts = [\"Semua\"] + sorted(df[\"cb1_line\"].dropna().unique().tolist())",
    "            sel_line  = st.multiselect(\"Line CB\", line_opts, default=[\"Semua\"])",
    "        else:",
    "            sel_line = [\"Semua\"]",
    "",
    "        sel_status = st.selectbox(\"Status\", [\"Semua\",\"Normal\",\"Defect\"])",
    "",
    "    st.divider()",
    "    st.markdown(\"\"\"",
    "    <div style='font-size:0.72rem; color:#475569; line-height:1.8;'>",
    "      <b>Cara Penggunaan:</b><br>",
    "      1. Upload CSV/Excel di sidebar kiri<br>",
    "      2. Gunakan filter untuk eksplorasi<br>",
    "      3. Navigasi tab untuk analisis<br>",
    "      4. Tab [4] Model untuk re-train<br>",
    "      5. Tab [3] Defect untuk penyebab",
    "    </div>",
    "    \"\"\", unsafe_allow_html=True)",
    "",
    "# ========================================================================",
    "# MAIN HEADER",
    "# ========================================================================",
    "st.markdown(\"\"\"",
    "<div style='padding:8px 0 20px'>",
    "  <div style='font-family:Syne,sans-serif; font-size:2rem; font-weight:800; color:#0f172a;'>",
    "    AI-Based Pharmaceutical Monitoring",
    "  </div>",
    "  <div style='color:#64748b; font-size:0.9rem; margin-top:4px;'>",
    "    Capstone Project PJK-GM016 &nbsp;*&nbsp; Pijak \u00d7 IBM SkillsBuild &nbsp;*&nbsp;",
    "    Granulasi Quality Control Dashboard",
    "  </div>",
    "</div>",
    "\"\"\", unsafe_allow_html=True)",
    "",
    "# -- No data state --------------------------------------------------------------",
    "if st.session_state.df is None:",
    "    st.markdown(\"\"\"",
    "    <div class='upload-hero'>",
    "      <div style='font-size:3rem; margin-bottom:12px;'>[>>]</div>",
    "      <div style='font-family:Syne,sans-serif; font-size:1.3rem; font-weight:700;",
    "                  color:#1e293b; margin-bottom:8px;'>",
    "        Upload Dataset untuk Memulai",
    "      </div>",
    "      <div style='color:#64748b; font-size:0.88rem; max-width:420px; margin:0 auto;'>",
    "        Upload file <b>Excel raw</b> (.xlsx/.xls) atau <b>CSV</b> (dataset_with_predictions / dataset_clean)<br>",
    "        melalui sidebar kiri untuk memulai monitoring dan analisis AI.",
    "      </div>",
    "    </div>",
    "    \"\"\", unsafe_allow_html=True)",
    "    st.stop()",
    "",
    "# -- Apply filters -------------------------------------------------------------",
    "df = st.session_state.df.copy()",
    "",
    "if \"Semua\" not in sel_bulan and \"cb1_bulan\" in df.columns:",
    "    df = df[df[\"cb1_bulan\"].isin(sel_bulan)]",
    "if \"Semua\" not in sel_line and \"cb1_line\" in df.columns:",
    "    df = df[df[\"cb1_line\"].isin(sel_line)]",
    "if sel_status != \"Semua\":",
    "    df = df[df[\"label_display\"] == sel_status]",
    "",
    "if df.empty:",
    "    st.warning(\"Tidak ada data setelah filter diterapkan.\")",
    "    st.stop()",
    "",
    "# -- KPI Cards -----------------------------------------------------------------",
    "render_kpis(df)",
    "",
    "# ========================================================================",
    "# TABS",
    "# ========================================================================",
    "tab_ov, tab_mon, tab_defect, tab_model, tab_cluster, tab_chat = st.tabs([",
    "    \"[1] Overview\",",
    "    \"[2] Monitoring\",",
    "    \"[3] Analisis Defect\",",
    "    \"[4] Model AI\",",
    "    \"[5] Clustering\",",
    "    \"[6] AI Chatbot\"",
    "])",
    "",
    "# ==============================",
    "# TAB 1 - OVERVIEW",
    "# ==============================",
    "with tab_ov:",
    "    # Distribusi",
    "    col1, col2, col3 = st.columns(3)",
    "",
    "    with col1:",
    "        st.markdown(\"<div class='section-title'>Proporsi Batch</div>\", unsafe_allow_html=True)",
    "        counts = df[\"label_display\"].value_counts()",
    "        fig = go.Figure(go.Pie(",
    "            labels=counts.index, values=counts.values,",
    "            marker_colors=[C_NORMAL if l==\"Normal\" else C_DEFECT for l in counts.index],",
    "            hole=0.52,",
    "            textinfo=\"percent+label\",",
    "            textfont=dict(size=13),",
    "        ))",
    "        fig.update_layout(**PLOT_LAYOUT, height=280, showlegend=False)",
    "        st.plotly_chart(fig, use_container_width=True)",
    "",
    "    with col2:",
    "        if \"cb1_bulan\" in df.columns:",
    "            st.markdown(\"<div class='section-title'>Defect per Bulan</div>\", unsafe_allow_html=True)",
    "            grp = df.groupby([\"cb1_bulan\",\"label_display\"]).size().reset_index(name=\"n\")",
    "            fig = px.bar(grp, x=\"cb1_bulan\", y=\"n\", color=\"label_display\",",
    "                         color_discrete_map={\"Normal\":C_NORMAL,\"Defect\":C_DEFECT},",
    "                         barmode=\"stack\")",
    "            fig.update_layout(**PLOT_LAYOUT, height=280, showlegend=True,",
    "                              xaxis_title=\"Bulan\", yaxis_title=\"Jumlah Batch\")",
    "            st.plotly_chart(fig, use_container_width=True)",
    "",
    "    with col3:",
    "        if \"cb1_line\" in df.columns:",
    "            st.markdown(\"<div class='section-title'>Defect per Line</div>\", unsafe_allow_html=True)",
    "            grp = df.groupby([\"cb1_line\",\"label_display\"]).size().reset_index(name=\"n\")",
    "            fig = px.bar(grp, x=\"cb1_line\", y=\"n\", color=\"label_display\",",
    "                         color_discrete_map={\"Normal\":C_NORMAL,\"Defect\":C_DEFECT},",
    "                         barmode=\"group\")",
    "            fig.update_layout(**PLOT_LAYOUT, height=280, showlegend=False,",
    "                              xaxis_title=\"Line\", yaxis_title=\"Jumlah Batch\")",
    "            st.plotly_chart(fig, use_container_width=True)",
    "",
    "    # Parameter distributions",
    "    if \"cb1_suhu_rata\" in df.columns and \"cb2_suhu_rata\" in df.columns:",
    "        st.markdown(\"<div class='section-title'>Distribusi Parameter Kritis (Normal vs Defect)</div>\",",
    "                    unsafe_allow_html=True)",
    "        params = []",
    "        if \"cb1_suhu_rata\" in df.columns: params.append((\"cb1_suhu_rata\",\"Suhu CB1 (\u00b0C)\"))",
    "        if \"cb2_suhu_rata\" in df.columns: params.append((\"cb2_suhu_rata\",\"Suhu CB2 (\u00b0C)\"))",
    "        if \"cb1_rata_ka\"   in df.columns: params.append((\"cb1_rata_ka\",  \"Kadar Air CB1 (%)\"))",
    "        if \"cb2_rata_ka\"   in df.columns: params.append((\"cb2_rata_ka\",  \"Kadar Air CB2 (%)\"))",
    "",
    "        cols = st.columns(min(4, len(params)))",
    "        for i, (col, label) in enumerate(params):",
    "            with cols[i % 4]:",
    "                fig = go.Figure()",
    "                for status, color in [(\"Normal\",C_NORMAL),(\"Defect\",C_DEFECT)]:",
    "                    vals = df[df[\"label_display\"]==status][col].dropna()",
    "                    fig.add_trace(go.Box(y=vals, name=status, marker_color=color,",
    "                                        boxmean=\"sd\", showlegend=(i==0)))",
    "                if \"suhu\" in col:",
    "                    fig.add_hline(y=75, line_dash=\"dot\", line_color=C_AMBER,",
    "                                  annotation_text=\"75\u00b0C\")",
    "                fig.update_layout(**PLOT_LAYOUT, height=260, title_text=label,",
    "                                  title_font_size=13)",
    "                st.plotly_chart(fig, use_container_width=True)",
    "",
    "    # Data table preview",
    "    st.markdown(\"<div class='section-title'>Tabel Data Batch</div>\", unsafe_allow_html=True)",
    "    show_cols = [\"row_no\",\"material_desc\",\"cb1_bulan\",\"cb1_line\",\"label_display\"]",
    "    if \"cb1_suhu_rata\" in df.columns: show_cols.append(\"cb1_suhu_rata\")",
    "    if \"cb1_pct_yield\" in df.columns: show_cols.append(\"cb1_pct_yield\")",
    "    if \"cb1_rata_ka\"   in df.columns: show_cols.append(\"cb1_rata_ka\")",
    "    if \"rf_proba_defect\" in df.columns: show_cols.append(\"rf_proba_defect\")",
    "    if \"consensus_label\" in df.columns: show_cols.append(\"consensus_label\")",
    "    show_cols = [c for c in show_cols if c in df.columns]",
    "",
    "    df_show = df[show_cols].copy()",
    "    df_show = df_show.rename(columns={",
    "        \"row_no\":\"#\",\"material_desc\":\"Produk\",\"cb1_bulan\":\"Bulan\",",
    "        \"cb1_line\":\"Line\",\"label_display\":\"Status Aktual\",",
    "        \"cb1_suhu_rata\":\"Suhu CB1\",\"cb1_pct_yield\":\"Yield CB1 %\",",
    "        \"cb1_rata_ka\":\"KA CB1\",\"rf_proba_defect\":\"P(Defect) RF\",",
    "        \"consensus_label\":\"Konsensus\"",
    "    })",
    "",
    "    def style_row(row):",
    "        is_def = (\"Defect\" in str(row.get(\"Status Aktual\",\"Normal\")))",
    "        bg = \"#fff1f2\" if is_def else \"#f0fdf4\"",
    "        return [f\"background:{bg}\"] * len(row)",
    "",
    "    st.dataframe(",
    "        df_show.style.apply(style_row, axis=1)",
    "               .format({c:\"{:.2f}\" for c in df_show.select_dtypes(\"float\").columns}, na_rep=\"-\"),",
    "        use_container_width=True, height=350",
    "    )",
    "",
    "# ==============================",
    "# TAB 2 - MONITORING",
    "# ==============================",
    "with tab_mon:",
    "    if \"cb1_pct_yield\" in df.columns:",
    "        st.markdown(\"<div class='section-title'>Monitoring Yield CB1 per Batch</div>\",",
    "                    unsafe_allow_html=True)",
    "        df_s = df.sort_values(\"row_no\").reset_index(drop=True)",
    "",
    "        fig = go.Figure()",
    "        for status, color, sym in [(\"Normal\",C_NORMAL,\"circle\"),(\"Defect\",C_DEFECT,\"x\")]:",
    "            sub = df_s[df_s[\"label_display\"]==status]",
    "            hov_data = [sub[c] if c in sub.columns else [\"-\"]*len(sub)",
    "                        for c in [\"material_desc\",\"cb1_bulan\",\"cb1_line\",\"defect_reasons\"]]",
    "            fig.add_trace(go.Scatter(",
    "                x=sub[\"row_no\"], y=sub[\"cb1_pct_yield\"],",
    "                mode=\"markers\",",
    "                name=status,",
    "                marker=dict(color=color, symbol=sym, size=10, opacity=0.85,",
    "                            line=dict(width=1, color=\"white\")),",
    "                customdata=np.stack([",
    "                    sub.get(\"material_desc\",[\"-\"]*len(sub)).fillna(\"-\"),",
    "                    sub.get(\"cb1_bulan\",[\"-\"]*len(sub)).fillna(\"-\"),",
    "                    sub.get(\"cb1_line\",[\"-\"]*len(sub)).fillna(\"-\"),",
    "                    sub.get(\"defect_reasons\",[\"-\"]*len(sub)).fillna(\"-\"),",
    "                ], axis=-1),",
    "                hovertemplate=(",
    "                    \"<b>Batch #%{x}</b><br>\"",
    "                    \"Yield CB1: <b>%{y:.2f}%</b><br>\"",
    "                    \"Produk: %{customdata[0]}<br>\"",
    "                    \"Bulan: %{customdata[1]} | Line: %{customdata[2]}<br>\"",
    "                    \"Alasan: %{customdata[3]}<extra></extra>\"",
    "                )",
    "            ))",
    "",
    "        # Moving average",
    "        ma = df_s[\"cb1_pct_yield\"].rolling(5, center=True).mean()",
    "        fig.add_trace(go.Scatter(",
    "            x=df_s[\"row_no\"], y=ma, mode=\"lines\",",
    "            name=\"Moving Avg (5)\", line=dict(color=C_BLUE, dash=\"dot\", width=2)",
    "        ))",
    "        fig.add_hline(y=99, line_dash=\"dash\", line_color=C_AMBER, line_width=1.5,",
    "                      annotation_text=\"Target 99%\", annotation_position=\"bottom right\")",
    "",
    "        fig.update_layout(**PLOT_LAYOUT, height=380,",
    "                          xaxis_title=\"No. Batch\", yaxis_title=\"Yield (%)\")",
    "        st.plotly_chart(fig, use_container_width=True)",
    "",
    "    # Anomaly score",
    "    if \"isolation_forest_score\" in df.columns:",
    "        st.markdown(\"<div class='section-title'>Anomaly Score - Isolation Forest</div>\",",
    "                    unsafe_allow_html=True)",
    "        df_if = df.sort_values(\"row_no\").reset_index(drop=True)",
    "        fig2 = go.Figure()",
    "        for status, color in [(\"Normal\",C_NORMAL),(\"Defect\",C_DEFECT)]:",
    "            sub = df_if[df_if[\"label_display\"]==status]",
    "            fig2.add_trace(go.Bar(",
    "                x=sub[\"row_no\"].astype(str), y=sub[\"isolation_forest_score\"],",
    "                name=status, marker_color=color, opacity=0.85,",
    "                hovertemplate=\"Batch #%{x}<br>Score: %{y:.4f}<extra></extra>\"",
    "            ))",
    "        fig2.add_hline(y=0, line_dash=\"dot\", line_color=\"#64748b\")",
    "        fig2.update_layout(**PLOT_LAYOUT, height=320,",
    "                           xaxis_title=\"No. Batch\", yaxis_title=\"Anomaly Score\",",
    "                           barmode=\"overlay\",",
    "                           annotations=[dict(x=0.99, y=0.03, xref=\"paper\", yref=\"paper\",",
    "                                             text=\"Semakin negatif = semakin anomali\",",
    "                                             showarrow=False, font=dict(color=\"#64748b\",size=11),",
    "                                             xanchor=\"right\")])",
    "        st.plotly_chart(fig2, use_container_width=True)",
    "",
    "    # Suhu vs Yield scatter",
    "    if \"cb1_suhu_rata\" in df.columns and \"cb1_pct_yield\" in df.columns:",
    "        st.markdown(\"<div class='section-title'>Suhu Rata-rata CB1 vs Yield CB1</div>\",",
    "                    unsafe_allow_html=True)",
    "        hover_extra = {}",
    "        if \"cb1_rata_ka\" in df.columns:",
    "            hover_extra[\"cb1_rata_ka\"] = True",
    "        fig3 = px.scatter(",
    "            df, x=\"cb1_suhu_rata\", y=\"cb1_pct_yield\",",
    "            color=\"label_display\",",
    "            color_discrete_map={\"Normal\":C_NORMAL,\"Defect\":C_DEFECT},",
    "            size=\"cb1_rata_ka\" if \"cb1_rata_ka\" in df.columns else None,",
    "            hover_data={\"row_no\":True,\"cb1_bulan\":True,\"cb1_line\":True,",
    "                        \"defect_reasons\":True} if \"defect_reasons\" in df.columns else {},",
    "            labels={\"cb1_suhu_rata\":\"Suhu CB1 (\u00b0C)\",\"cb1_pct_yield\":\"Yield CB1 (%)\"},",
    "        )",
    "        fig3.add_vline(x=75, line_dash=\"dot\", line_color=C_AMBER,",
    "                       annotation_text=\"Batas 75\u00b0C\")",
    "        fig3.add_hline(y=99, line_dash=\"dot\", line_color=C_BLUE,",
    "                       annotation_text=\"Target 99%\")",
    "        fig3.update_layout(**PLOT_LAYOUT, height=400)",
    "        st.plotly_chart(fig3, use_container_width=True)",
    "",
    "# ==============================",
    "# TAB 3 - ANALISIS DEFECT",
    "# ==============================",
    "with tab_defect:",
    "    df_def = df[df[\"is_defect\"] == 1].copy()",
    "",
    "    if df_def.empty:",
    "        st.info(\"Tidak ada batch defect dalam data yang difilter.\")",
    "    else:",
    "        col_l, col_r = st.columns([1, 1])",
    "",
    "        with col_l:",
    "            st.markdown(\"<div class='section-title'>Frekuensi Penyebab Defect</div>\",",
    "                        unsafe_allow_html=True)",
    "            if \"defect_reasons\" in df_def.columns:",
    "                rc = parse_reasons(df_def[\"defect_reasons\"]).head(15)",
    "                if not rc.empty:",
    "                    fig = go.Figure(go.Bar(",
    "                        x=rc.values, y=rc.index, orientation=\"h\",",
    "                        marker=dict(color=rc.values, colorscale=\"Reds\", showscale=False),",
    "                        text=rc.values, textposition=\"outside\"",
    "                    ))",
    "                    _layout = {**PLOT_LAYOUT,",
    "                               \"yaxis\": {**PLOT_LAYOUT.get(\"yaxis\", {}),",
    "                                         \"autorange\": \"reversed\",",
    "                                         \"gridcolor\": \"#e2e8f0\"}}",
    "                    _layout[\"margin\"] = dict(l=280, r=30, t=20, b=30)",
    "                    fig.update_layout(**_layout, height=460,",
    "                                      xaxis_title=\"Jumlah Kemunculan\")",
    "",
    "                    st.plotly_chart(fig, use_container_width=True)",
    "",
    "        with col_r:",
    "            st.markdown(\"<div class='section-title'>Probabilitas Defect (RF & GB)</div>\",",
    "                        unsafe_allow_html=True)",
    "            if \"rf_proba_defect\" in df.columns:",
    "                fig2 = make_subplots(rows=2, cols=1,",
    "                                     subplot_titles=[\"Random Forest\",\"Gradient Boosting\"])",
    "                for row_idx, (col_p, cname) in enumerate(",
    "                    [(\"rf_proba_defect\",\"RF\"),(\"gb_proba_defect\",\"GB\")], 1",
    "                ):",
    "                    if col_p in df.columns:",
    "                        for status, color in [(\"Normal\",C_NORMAL),(\"Defect\",C_DEFECT)]:",
    "                            sub = df[df[\"label_display\"]==status]",
    "                            fig2.add_trace(go.Histogram(",
    "                                x=sub[col_p], name=f\"{cname} {status}\",",
    "                                marker_color=color, opacity=0.7, nbinsx=15,",
    "                                showlegend=(row_idx==1)",
    "                            ), row=row_idx, col=1)",
    "                        fig2.add_vline(x=0.5, line_dash=\"dot\",",
    "                                       line_color=C_AMBER, row=row_idx, col=1)",
    "                fig2.update_layout(**PLOT_LAYOUT, height=460, barmode=\"overlay\")",
    "                st.plotly_chart(fig2, use_container_width=True)",
    "",
    "        # Reason cards per batch",
    "        st.markdown(\"<div class='section-title'>Detail Penyebab Defect per Batch</div>\",",
    "                    unsafe_allow_html=True)",
    "",
    "        show_detail_cols = [\"row_no\",\"material_desc\",\"cb1_bulan\",\"cb1_line\",",
    "                            \"cb1_suhu_rata\",\"cb1_pct_yield\",\"defect_reasons\"]",
    "        show_detail_cols = [c for c in show_detail_cols if c in df_def.columns]",
    "",
    "        n_shown = 0",
    "        for _, row in df_def[show_detail_cols].iterrows():",
    "            reasons_raw = str(row.get(\"defect_reasons\",\"-\"))",
    "            reasons_list = [r.strip() for r in reasons_raw.split(\"|\")",
    "                            if r.strip() and r.strip() != \"-\"]",
    "",
    "            batch_id = row.get(\"row_no\",\"?\")",
    "            produk   = row.get(\"material_desc\",\"-\")",
    "            bulan    = row.get(\"cb1_bulan\",\"-\")",
    "            line     = row.get(\"cb1_line\",\"-\")",
    "            suhu     = f\"{row['cb1_suhu_rata']:.1f}\u00b0C\" if \"cb1_suhu_rata\" in row and pd.notna(row.get(\"cb1_suhu_rata\")) else \"-\"",
    "            yld      = f\"{row['cb1_pct_yield']:.2f}%\" if \"cb1_pct_yield\" in row and pd.notna(row.get(\"cb1_pct_yield\")) else \"-\"",
    "",
    "            badges  = \" \".join([f\"<span class='reason-badge'>[!] {r}</span>\" for r in reasons_list])",
    "            if not badges:",
    "                badges = \"<span class='reason-badge' style='background:#fef3c7;color:#92400e;'>Tidak ada detail</span>\"",
    "",
    "            st.markdown(f\"\"\"",
    "            <div class='reason-card'>",
    "              <div style='display:flex; align-items:center; gap:10px; flex-wrap:wrap;'>",
    "                <b style='font-family:Syne,sans-serif;'>Batch #{batch_id}</b>",
    "                <span style='color:#64748b; font-size:0.8rem;'>",
    "                  {produk} &nbsp;*&nbsp; {bulan} &nbsp;*&nbsp; {line}",
    "                  &nbsp;*&nbsp; Suhu: {suhu} &nbsp;*&nbsp; Yield: {yld}",
    "                </span>",
    "              </div>",
    "              <div class='reason-text' style='margin-top:8px;'>{badges}</div>",
    "            </div>",
    "            \"\"\", unsafe_allow_html=True)",
    "            n_shown += 1",
    "            if n_shown >= 30:",
    "                st.caption(f\"Menampilkan 30 dari {len(df_def)} batch defect.\")",
    "                break",
    "",
    "        # Heatmap korelasi",
    "        crit = [c for c in [\"cb1_suhu_rata\",\"cb2_suhu_rata\",\"cb1_rata_ka\",\"cb2_rata_ka\",",
    "                             \"cb1_pct_yield\",\"ck1_pct_yield\",\"cb1_suhu_std\",\"ka_ekstrem_count\",",
    "                             \"is_defect\"]",
    "                if c in df.columns]",
    "        if len(crit) >= 3:",
    "            st.markdown(\"<div class='section-title'>Heatmap Korelasi Parameter Kritis</div>\",",
    "                        unsafe_allow_html=True)",
    "            corr = df[crit].corr()",
    "            fig3 = go.Figure(go.Heatmap(",
    "                z=corr.values, x=corr.columns, y=corr.index,",
    "                colorscale=\"RdBu_r\", zmid=0,",
    "                text=corr.values.round(2), texttemplate=\"%{text}\",",
    "                textfont=dict(size=10)",
    "            ))",
    "            fig3.update_layout(**PLOT_LAYOUT, height=400)",
    "            st.plotly_chart(fig3, use_container_width=True)",
    "",
    "# ==============================",
    "# TAB 4 - MODEL AI",
    "# ==============================",
    "with tab_model:",
    "    st.markdown(\"<div class='section-title'>Re-train Model AI</div>\", unsafe_allow_html=True)",
    "",
    "    col_btn, col_info = st.columns([1,2])",
    "    with col_btn:",
    "        run_ml = st.button(\"[RUN] Jalankan Pipeline ML\", type=\"primary\",",
    "                           use_container_width=True)",
    "    with col_info:",
    "        st.caption(\"Klik untuk melatih ulang Random Forest, Gradient Boosting, dan \"",
    "                   \"Isolation Forest menggunakan dataset yang diupload.\")",
    "",
    "    if run_ml:",
    "        with st.spinner(\"Melatih model... ini mungkin membutuhkan beberapa detik.\"):",
    "            df_ml, fi_df, feat_cols = run_ml_pipeline(st.session_state.df.copy())",
    "        if fi_df is not None:",
    "            st.session_state.df = df_ml",
    "            df = df_ml.copy()",
    "            # apply filter again",
    "            if \"Semua\" not in sel_bulan and \"cb1_bulan\" in df.columns:",
    "                df = df[df[\"cb1_bulan\"].isin(sel_bulan)]",
    "            if \"Semua\" not in sel_line and \"cb1_line\" in df.columns:",
    "                df = df[df[\"cb1_line\"].isin(sel_line)]",
    "            if sel_status != \"Semua\":",
    "                df = df[df[\"label_display\"] == sel_status]",
    "            st.success(\"[OK] Model berhasil dilatih!\")",
    "        else:",
    "            st.error(\"Pipeline gagal -- pastikan kolom is_defect tersedia.\")",
    "            fi_df = None",
    "",
    "    # -- Metrics --------------------------------------------------------",
    "    pred_cols = {",
    "        \"Random Forest\"    : \"rf_pred\",",
    "        \"Gradient Boosting\": \"gb_pred\",",
    "        \"Isolation Forest\" : \"if_pred\",",
    "        \"Konsensus\"        : \"model_consensus_pred\",",
    "    }",
    "    available_preds = {k:v for k,v in pred_cols.items() if v in df.columns}",
    "",
    "    if \"is_defect\" in df.columns and available_preds:",
    "        st.markdown(\"<div class='section-title'>Metrik Evaluasi Model</div>\",",
    "                    unsafe_allow_html=True)",
    "        y_true = df[\"is_defect\"].astype(int)",
    "        rows = []",
    "        for name, col in available_preds.items():",
    "            yp = df[col].astype(int)",
    "            rows.append({",
    "                \"Model\"    : name,",
    "                \"Accuracy\" : accuracy_score(y_true, yp),",
    "                \"Precision\": precision_score(y_true, yp, zero_division=0),",
    "                \"Recall\"   : recall_score(y_true, yp, zero_division=0),",
    "                \"F1-Score\" : f1_score(y_true, yp, zero_division=0),",
    "            })",
    "        met_df = pd.DataFrame(rows)",
    "",
    "        fig_m = go.Figure()",
    "        colors_m = [C_BLUE, C_NORMAL, C_AMBER, C_PURPLE]",
    "        for metric, color in zip([\"Accuracy\",\"Precision\",\"Recall\",\"F1-Score\"], colors_m):",
    "            fig_m.add_trace(go.Bar(",
    "                x=met_df[\"Model\"], y=met_df[metric], name=metric,",
    "                marker_color=color,",
    "                text=met_df[metric].round(3), textposition=\"outside\"",
    "            ))",
    "        _layout_m = {**PLOT_LAYOUT,",
    "                     \"yaxis\": {**PLOT_LAYOUT.get(\"yaxis\", {}),",
    "                               \"range\": [0, 1.18], \"gridcolor\": \"#e2e8f0\"}}",
    "        fig_m.update_layout(**_layout_m, height=360, barmode=\"group\")",
    "        st.plotly_chart(fig_m, use_container_width=True)",
    "",
    "        # Confusion matrices",
    "        st.markdown(\"<div class='section-title'>Confusion Matrix</div>\",",
    "                    unsafe_allow_html=True)",
    "        n_mod = len(available_preds)",
    "        fig_cm = make_subplots(rows=1, cols=n_mod,",
    "                               subplot_titles=list(available_preds.keys()))",
    "        for idx, (name, col) in enumerate(available_preds.items(), 1):",
    "            cm = confusion_matrix(y_true, df[col].astype(int))",
    "            lbl = [[\"TN\",\"FP\"],[\"FN\",\"TP\"]]",
    "            ann = [[f\"{lbl[r][c]}<br>{cm[r,c]}\" for c in range(2)] for r in range(2)]",
    "            fig_cm.add_trace(go.Heatmap(",
    "                z=cm, text=ann, texttemplate=\"%{text}\",",
    "                colorscale=\"RdYlGn\", showscale=False,",
    "                x=[\"Pred:Normal\",\"Pred:Defect\"],",
    "                y=[\"Act:Normal\",\"Act:Defect\"],",
    "                xgap=2, ygap=2",
    "            ), row=1, col=idx)",
    "        fig_cm.update_layout(**PLOT_LAYOUT, height=300,",
    "                             font=dict(size=11, color=\"#374151\"))",
    "        st.plotly_chart(fig_cm, use_container_width=True)",
    "",
    "    # -- Feature Importance ---------------------------------------------",
    "    if \"rf_pred\" in df.columns:",
    "        st.markdown(\"<div class='section-title'>Feature Importance - Random Forest</div>\",",
    "                    unsafe_allow_html=True)",
    "        # Re-compute feature importance",
    "        _, fi_fresh, _ = run_ml_pipeline(st.session_state.df.copy())",
    "        if fi_fresh is not None:",
    "            top20 = fi_fresh.head(20)",
    "            fig_fi = go.Figure(go.Bar(",
    "                x=top20[\"importance\"], y=top20[\"feature\"],",
    "                orientation=\"h\",",
    "                marker=dict(color=top20[\"importance\"],",
    "                            colorscale=\"Blues\", showscale=True,",
    "                            colorbar=dict(title=\"Importance\", thickness=12)),",
    "                hovertemplate=\"%{y}: %{x:.4f}<extra></extra>\"",
    "            ))",
    "            _layout_fi = {**PLOT_LAYOUT,",
    "                          \"yaxis\": {**PLOT_LAYOUT.get(\"yaxis\", {}),",
    "                                    \"autorange\": \"reversed\", \"gridcolor\": \"#e2e8f0\"}}",
    "            _layout_fi[\"margin\"] = dict(l=250, r=60, t=20, b=30)",
    "            fig_fi.update_layout(**_layout_fi, height=520,",
    "                                  xaxis_title=\"Importance Score\")",
    "",
    "            st.plotly_chart(fig_fi, use_container_width=True)",
    "",
    "# ==============================",
    "# TAB 5 - CLUSTERING",
    "# ==============================",
    "with tab_cluster:",
    "    exclude_kw_cl = [\"pred\",\"proba\",\"probability\",\"status\",\"label\",\"consensus\",",
    "                     \"defect_reasons\",\"reasons\",\"enc\",\"num\",\"row_no\"]",
    "    exclude_cols_cl = [\"is_defect\",\"label_display\",\"pred_label\",",
    "                       \"material_desc\",\"cb1_bulan\",\"cb2_bulan\",",
    "                       \"cb1_line\",\"cb2_line\",\"cb1_shift\",\"cb2_shift\",",
    "                       \"ck1_keterangan\",\"ck2_keterangan\",",
    "                       \"lapis1_warna\",\"lapis2_warna\"]",
    "    feat_cl = [c for c in df.columns",
    "               if c not in exclude_cols_cl",
    "               and not any(k in c.lower() for k in exclude_kw_cl)",
    "               and df[c].dtype in [np.float64, np.int64, float, int]]",
    "",
    "    if len(feat_cl) < 3:",
    "        st.info(\"Fitur numerik tidak cukup untuk clustering.\")",
    "    else:",
    "        df_cl = df[feat_cl].copy().apply(lambda s: s.fillna(s.median()))",
    "        scaler_cl = StandardScaler()",
    "        Xs_cl = scaler_cl.fit_transform(df_cl)",
    "",
    "        col_el, col_n = st.columns([2,1])",
    "        with col_n:",
    "            n_clusters = st.slider(\"Jumlah Cluster (k)\", 2, 6, 3)",
    "",
    "        # Elbow",
    "        from sklearn.cluster import KMeans as KM",
    "        inertias = {}",
    "        for k in range(2, 8):",
    "            km_ = KM(n_clusters=k, random_state=42, n_init=10)",
    "            km_.fit(Xs_cl)",
    "            inertias[k] = km_.inertia_",
    "",
    "        with col_el:",
    "            st.markdown(\"<div class='section-title'>Elbow Method</div>\",",
    "                        unsafe_allow_html=True)",
    "            fig_el = go.Figure(go.Scatter(",
    "                x=list(inertias.keys()), y=list(inertias.values()),",
    "                mode=\"lines+markers\",",
    "                marker=dict(color=C_BLUE, size=9),",
    "                line=dict(color=C_BLUE, width=2)",
    "            ))",
    "            fig_el.add_vline(x=n_clusters, line_dash=\"dot\", line_color=C_DEFECT,",
    "                             annotation_text=f\"k={n_clusters}\")",
    "            fig_el.update_layout(**PLOT_LAYOUT, height=260,",
    "                                  xaxis_title=\"k\", yaxis_title=\"Inertia\")",
    "            st.plotly_chart(fig_el, use_container_width=True)",
    "",
    "        # Fit KMeans",
    "        km = KM(n_clusters=n_clusters, random_state=42, n_init=10)",
    "        df[\"cluster\"] = km.fit_predict(Xs_cl)",
    "",
    "        # PCA 2D",
    "        pca = PCA(n_components=2, random_state=42)",
    "        pc  = pca.fit_transform(Xs_cl)",
    "        df[\"pca1\"] = pc[:,0]",
    "        df[\"pca2\"] = pc[:,1]",
    "        var = pca.explained_variance_ratio_",
    "",
    "        st.markdown(\"<div class='section-title'>Visualisasi Clustering (PCA 2D)</div>\",",
    "                    unsafe_allow_html=True)",
    "        col_c1, col_c2 = st.columns(2)",
    "",
    "        cl_colors = [C_BLUE, C_NORMAL, C_AMBER, C_PURPLE, \"#ec4899\", \"#14b8a6\"]",
    "        with col_c1:",
    "            fig_cl = go.Figure()",
    "            for cl in sorted(df[\"cluster\"].unique()):",
    "                sub = df[df[\"cluster\"]==cl]",
    "                fig_cl.add_trace(go.Scatter(",
    "                    x=sub[\"pca1\"], y=sub[\"pca2\"], mode=\"markers\",",
    "                    name=f\"Cluster {cl}\",",
    "                    marker=dict(color=cl_colors[cl], size=9, opacity=0.8,",
    "                                line=dict(width=0.5,color=\"white\")),",
    "                    hovertemplate=f\"Cluster {cl}<br>PC1: %{{x:.2f}}<br>PC2: %{{y:.2f}}<extra></extra>\"",
    "                ))",
    "            fig_cl.update_layout(**PLOT_LAYOUT, height=380,",
    "                                  xaxis_title=f\"PC1 ({var[0]:.1%})\",",
    "                                  yaxis_title=f\"PC2 ({var[1]:.1%})\",",
    "                                  title_text=\"Warna: Cluster\")",
    "            st.plotly_chart(fig_cl, use_container_width=True)",
    "",
    "        with col_c2:",
    "            fig_cl2 = go.Figure()",
    "            for status, color, sym in [(\"Normal\",C_NORMAL,\"circle\"),(\"Defect\",C_DEFECT,\"x\")]:",
    "                sub = df[df[\"label_display\"]==status]",
    "                fig_cl2.add_trace(go.Scatter(",
    "                    x=sub[\"pca1\"], y=sub[\"pca2\"], mode=\"markers\",",
    "                    name=status,",
    "                    marker=dict(color=color, symbol=sym, size=9, opacity=0.85,",
    "                                line=dict(width=0.5,color=\"white\")),",
    "                ))",
    "            fig_cl2.update_layout(**PLOT_LAYOUT, height=380,",
    "                                   xaxis_title=f\"PC1 ({var[0]:.1%})\",",
    "                                   yaxis_title=f\"PC2 ({var[1]:.1%})\",",
    "                                   title_text=\"Warna: Status Aktual\")",
    "            st.plotly_chart(fig_cl2, use_container_width=True)",
    "",
    "        # Cluster summary",
    "        st.markdown(\"<div class='section-title'>Profil Cluster</div>\",",
    "                    unsafe_allow_html=True)",
    "        cluster_sum = (",
    "            df.groupby(\"cluster\")[\"is_defect\"]",
    "              .agg(Total=\"count\", Defect=\"sum\")",
    "              .assign(Normal=lambda x: x[\"Total\"]-x[\"Defect\"],",
    "                      Defect_Rate=lambda x: (x[\"Defect\"]/x[\"Total\"]*100).round(1))",
    "        )",
    "        st.dataframe(cluster_sum.style",
    "                     .format({\"Defect_Rate\":\"{:.1f}%\"})",
    "                     .background_gradient(subset=[\"Defect_Rate\"], cmap=\"Reds\"),",
    "                     use_container_width=True)",
    "",
    "",
    "# ==============================",
    "# TAB 6 - AI CHATBOT",
    "# ==============================",
    "with tab_chat:",
    "    st.markdown(\"<div class='section-title'>[AI] Chatbot Analisis Granulasi</div>\", unsafe_allow_html=True)",
    "",
    "    # Session state for chat",
    "    if 'chat_messages' not in st.session_state:",
    "        st.session_state.chat_messages = []",
    "",
    "    # Build data summary for context",
    "    def build_data_context(df):",
    "        total = len(df)",
    "        n_def = int(df[\'is_defect\'].sum()) if \'is_defect\' in df.columns else 0",
    "        dr = n_def/total*100 if total else 0",
    "        avg_suhu1 = df[\'cb1_suhu_rata\'].mean() if \'cb1_suhu_rata\' in df.columns else None",
    "        avg_suhu2 = df[\'cb2_suhu_rata\'].mean() if \'cb2_suhu_rata\' in df.columns else None",
    "        avg_yield1 = df[\'cb1_pct_yield\'].mean() if \'cb1_pct_yield\' in df.columns else None",
    "        avg_ka1 = df[\'cb1_rata_ka\'].mean() if \'cb1_rata_ka\' in df.columns else None",
    "        defect_batches = df[df[\'is_defect\']==1][\'defect_reasons\'].dropna().tolist() if \'defect_reasons\' in df.columns else []",
    "        import re as _re",
    "        reason_counter = {}",
    "        for r in defect_batches:",
    "            for part in str(r).split(\'|\'):",
    "                part = _re.sub(r\'\\\\(.*?\\\\)\',\'\',part).strip()",
    "                if part and part not in (\'-\',\'nan\',\'\'):",
    "                    reason_counter[part] = reason_counter.get(part,0)+1",
    "        top_reasons = sorted(reason_counter, key=reason_counter.get, reverse=True)[:5]",
    "        s1 = str(round(avg_suhu1,2))+\'°C\' if avg_suhu1 else \'N/A\'",
    "        s2 = str(round(avg_suhu2,2))+\'°C\' if avg_suhu2 else \'N/A\'",
    "        y1 = str(round(avg_yield1,2))+\'%\' if avg_yield1 else \'N/A\'",
    "        k1 = str(round(avg_ka1,3))+\'%\' if avg_ka1 else \'N/A\'",
    "        reasons_str = \'\\n\'.join([\'  * \'+r for r in top_reasons]) if top_reasons else \'N/A\'",
    "        ctx = (\'DATA SUMMARY (dataset_with_predictions.csv):\\n\'",
    "                + \'- Total batch: \' + str(total) + \'\\n\'",
    "                + \'- Batch defect: \' + str(n_def) + \' (\' + str(round(dr,1)) + \'%)\\n\'",
    "                + \'- Batch normal: \' + str(total-n_def) + \' (\' + str(round(100-dr,1)) + \'%)\\n\'",
    "                + \'- Rata-rata suhu CB1: \' + s1 + \'\\n\'",
    "                + \'- Rata-rata suhu CB2: \' + s2 + \'\\n\'",
    "                + \'- Rata-rata yield CB1: \' + y1 + \'\\n\'",
    "                + \'- Rata-rata kadar air CB1: \' + k1 + \'\\n\'",
    "                + \'- Top penyebab defect:\\n\' + reasons_str + \'\\n\'",
    "                + \'- Batas normal: suhu <= 75C, yield >= 99%\\n\'",
    "                + \'- Proyek: AI-Based Pharmaceutical Monitoring (PJK-GM016)\\n\'",
    "                + \'- Model ML: Random Forest, Gradient Boosting, Isolation Forest\')",
    "        return ctx",
    "",
    "    # Chat UI",
    "    st.markdown(\"\"\"<style>",
    "    .chat-container { max-height: 480px; overflow-y: auto; padding: 12px; background: #f8fafc; border-radius: 16px; border: 1px solid #e2e8f0; margin-bottom: 12px; }",
    "    .chat-msg-user { display: flex; justify-content: flex-end; margin-bottom: 10px; }",
    "    .chat-msg-user .bubble { background: linear-gradient(135deg,#3b82f6,#1d4ed8); color: white; padding: 10px 16px; border-radius: 18px 18px 4px 18px; max-width: 75%; font-size: 0.88rem; line-height: 1.5; }",
    "    .chat-msg-bot { display: flex; justify-content: flex-start; margin-bottom: 10px; align-items: flex-start; gap: 8px; }",
    "    .chat-msg-bot .avatar { background: linear-gradient(135deg,#8b5cf6,#4c1d95); color: white; width: 32px; height: 32px; border-radius: 50%; display: flex; align-items: center; justify-content: center; font-size: 0.8rem; font-weight: 700; flex-shrink: 0; }",
    "    .chat-msg-bot .bubble { background: white; color: #1e293b; padding: 10px 16px; border-radius: 4px 18px 18px 18px; max-width: 80%; font-size: 0.88rem; line-height: 1.5; box-shadow: 0 2px 8px rgba(0,0,0,0.06); border: 1px solid #e2e8f0; }",
    "    .chat-empty { text-align: center; color: #94a3b8; padding: 40px 20px; font-size: 0.9rem; }",
    "    </style>\"\"\", unsafe_allow_html=True)",
    "",
    "    # Render chat history",
    "    chat_html = '<div class=\"chat-container\" id=\"chatbox\">'",
    "    if not st.session_state.chat_messages:",
    "        chat_html += '<div class=\"chat-empty\">Tanya saya tentang data granulasi kapsul ini!<br><br>'",
    "        chat_html += '<small>Contoh: \"Berapa batch yang defect?\", \"Apa penyebab utama defect?\", \"Rekomendasi untuk meningkatkan yield?\"</small></div>'",
    "    for msg in st.session_state.chat_messages:",
    "        role = msg['role']",
    "        content = msg['content'].replace(chr(10), '<br>')",
    "        if role == 'user':",
    "            chat_html += f'<div class=\"chat-msg-user\"><div class=\"bubble\">{content}</div></div>'",
    "        else:",
    "            chat_html += f'<div class=\"chat-msg-bot\"><div class=\"avatar\">AI</div><div class=\"bubble\">{content}</div></div>'",
    "    chat_html += '</div>'",
    "    st.markdown(chat_html, unsafe_allow_html=True)",
    "",
    "    # Input",
    "    col_inp, col_btn, col_clr = st.columns([6,1,1])",
    "    with col_inp:",
    "        user_input = st.text_input(\"\", placeholder=\"Tanya sesuatu tentang data granulasi...\", key=\"chat_input\", label_visibility=\"collapsed\")",
    "    with col_btn:",
    "        send = st.button(\"Kirim\", use_container_width=True, type=\"primary\")",
    "    with col_clr:",
    "        clear = st.button(\"Reset\", use_container_width=True)",
    "",
    "    if clear:",
    "        st.session_state.chat_messages = []",
    "        st.rerun()",
    "",
    "    if send and user_input.strip():",
    "        st.session_state.chat_messages.append({'role':'user','content':user_input})",
    "        data_ctx = build_data_context(df)",
    "        system_prompt = (",
    "            'Kamu adalah AI analyst sistem monitoring granulasi farmasi.\\n'",
    "            'Jawab dalam Bahasa Indonesia, ringkas dan actionable.\\n'",
    "            'Fokus pada: penyebab defect, rekomendasi perbaikan, interpretasi data.\\n'",
    "            'Jika ditanya di luar konteks, tetap relevan dengan industri farmasi.\\n'",
    "            'Gunakan data berikut:\\n' + data_ctx",
    "        )",
    "        history = [{'role':m['role'],'content':m['content']} for m in st.session_state.chat_messages]",
    "        import urllib.request, urllib.error, json as _json",
    "        payload = _json.dumps({",
    "            'model': 'claude-sonnet-4-20250514',",
    "            'max_tokens': 1000,",
    "            'system': system_prompt,",
    "            'messages': history",
    "        }).encode()",
    "        api_key = st.secrets.get('ANTHROPIC_API_KEY','')",
    "        if not api_key:",
    "            reply = 'Error: ANTHROPIC_API_KEY belum diset di Streamlit secrets. Tambahkan di .streamlit/secrets.toml: ANTHROPIC_API_KEY=\"sk-ant-...\"'",
    "        else:",
    "            try:",
    "                req = urllib.request.Request(",
    "                    'https://api.anthropic.com/v1/messages',",
    "                    data=payload,",
    "                    headers={'Content-Type':'application/json','x-api-key':api_key,'anthropic-version':'2023-06-01'},",
    "                    method='POST'",
    "                )",
    "                with urllib.request.urlopen(req, timeout=30) as resp:",
    "                    result = _json.loads(resp.read().decode())",
    "                reply = result['content'][0]['text']",
    "            except Exception as e:",
    "                reply = f'Gagal: {str(e)}'",
    "        st.session_state.chat_messages.append({'role':'assistant','content':reply})",
    "        st.rerun()",
    "",
    "    # Quick question buttons",
    "    st.markdown(\"**Pertanyaan cepat:**\")",
    "    qcols = st.columns(3)",
    "    quick_qs = [",
    "        \"Berapa batch yang defect dan apa penyebab utamanya?\",",
    "        \"Rekomendasi untuk menurunkan defect rate?\",",
    "        \"Bagaimana performa model ML dalam proyek ini?\",",
    "    ]",
    "    for i, q in enumerate(quick_qs):",
    "        with qcols[i]:",
    "            if st.button(q, key=f'quick_{i}', use_container_width=True):",
    "                st.session_state.chat_messages.append({'role':'user','content':q})",
    "                data_ctx = build_data_context(df)",
    "                system_prompt = (",
    "                    'Kamu adalah AI analyst sistem monitoring granulasi farmasi.\\n'",
    "                    'Jawab dalam Bahasa Indonesia, ringkas dan actionable.\\n'",
    "                    'Gunakan data berikut:\\n' + data_ctx",
    "                )",
    "                history = [{'role':m['role'],'content':m['content']} for m in st.session_state.chat_messages]",
    "                import urllib.request as _ur, json as _j",
    "                payload = _j.dumps({'model':'claude-sonnet-4-20250514','max_tokens':800,'system':system_prompt,'messages':history}).encode()",
    "                api_key = st.secrets.get('ANTHROPIC_API_KEY','')",
    "                if not api_key:",
    "                    reply = 'Error: ANTHROPIC_API_KEY belum diset di secrets.'",
    "                else:",
    "                    try:",
    "                        req = _ur.Request('https://api.anthropic.com/v1/messages',data=payload,headers={'Content-Type':'application/json','x-api-key':api_key,'anthropic-version':'2023-06-01'},method='POST')",
    "                        with _ur.urlopen(req,timeout=30) as resp: result = _j.loads(resp.read().decode())",
    "                        reply = result['content'][0]['text']",
    "                    except Exception as e:",
    "                        reply = f'Gagal: {str(e)}'",
    "                st.session_state.chat_messages.append({'role':'assistant','content':reply})",
    "                st.rerun()",
    "",
    "# -- Footer ---------------------------------------------------------------------",
    "st.markdown(\"\"\"",
    "<div class='footer'>",
    "  [APP] AI-Based Pharmaceutical Monitoring System &nbsp;*&nbsp;",
    "  Capstone Project <b>PJK-GM016</b> &nbsp;*&nbsp;",
    "  Pijak \u00d7 IBM SkillsBuild &nbsp;*&nbsp;",
    "  Dibuat dengan Streamlit + Plotly",
    "</div>",
    "\"\"\", unsafe_allow_html=True)",
    "",
]

output_path = 'app.py'
with open(output_path, 'w', encoding='utf-8') as fout:
    fout.write('\n'.join(APP_LINES))

size_kb = os.path.getsize(output_path) / 1024
print(f'app.py berhasil ditulis: {output_path}')
print(f'Ukuran: {size_kb:.1f} KB')
print(f'Baris kode: {len(APP_LINES)}')
print()
print('Jalankan dengan:')
print('  streamlit run app.py')


app.py berhasil ditulis: app.py
Ukuran: 52.5 KB
Baris kode: 1146

Jalankan dengan:
  streamlit run app.py


## 3. Verifikasi Sintaks `app.py`


In [4]:
import ast

with open('app.py', encoding='utf-8') as f:
    source = f.read()

try:
    ast.parse(source)
    lines = source.count('\n')
    print('Sintaks Python valid!')
    print(f'Total baris  : {lines}')
    print(f'Ukuran file  : {len(source)/1024:.1f} KB')
except SyntaxError as e:
    print(f'Syntax error: {e}')


Sintaks Python valid!
Total baris  : 1145
Ukuran file  : 51.4 KB


## 4. Jalankan Dashboard


In [5]:
print('=' * 55)
print('  CARA MENJALANKAN DASHBOARD')
print('=' * 55)
print()
print('Opsi A (Disarankan) - Terminal:')
print('  streamlit run app.py')
print()
print('Opsi B - Dari notebook (uncomment baris di bawah):')
print()

# import subprocess, sys, time
# proc = subprocess.Popen(
#     [sys.executable, '-m', 'streamlit', 'run', 'app.py',
#      '--server.headless', 'true'],
#     stdout=subprocess.PIPE, stderr=subprocess.PIPE
# )
# time.sleep(3)
# print('Dashboard berjalan di: http://localhost:8501')

print('Setelah berjalan, buka browser:')
print('  http://localhost:8501')
print()
print('Langkah penggunaan:')
print('  1. Upload dataset_with_predictions.csv via sidebar kiri')
print('  2. Gunakan filter Bulan / Line / Status untuk eksplorasi')
print('  3. Navigasi antar tab (Overview, Monitoring, Defect, Model, Cluster)')
print('  4. Tab Model AI: klik Jalankan Pipeline ML untuk re-train model')
print('  5. Tab Analisis Defect: lihat penyebab defect per batch')


  CARA MENJALANKAN DASHBOARD

Opsi A (Disarankan) - Terminal:
  streamlit run app.py

Opsi B - Dari notebook (uncomment baris di bawah):

Setelah berjalan, buka browser:
  http://localhost:8501

Langkah penggunaan:
  1. Upload dataset_with_predictions.csv via sidebar kiri
  2. Gunakan filter Bulan / Line / Status untuk eksplorasi
  3. Navigasi antar tab (Overview, Monitoring, Defect, Model, Cluster)
  4. Tab Model AI: klik Jalankan Pipeline ML untuk re-train model
  5. Tab Analisis Defect: lihat penyebab defect per batch


## 5. Checklist Testing


In [6]:
print('================================================================')
print('  CHECKLIST ALPHA TESTING (Internal Tim)')
print('================================================================')
print('  [ ] Dashboard terbuka di http://localhost:8501')
print('  [ ] Upload CSV berhasil (dataset_with_predictions.csv)')
print('  [ ] KPI cards tampil dengan angka benar')
print('  [ ] Filter Bulan / Line / Status berfungsi')
print('  [ ] Tab Overview: chart distribusi dan tabel data tampil')
print('  [ ] Tab Monitoring: scatter yield + anomaly score tampil')
print('  [ ] Tab Analisis Defect: reason cards muncul per batch')
print('  [ ] Tab Model AI: tombol Re-train berhasil dijalankan')
print('  [ ] Tab Clustering: PCA scatter dan profil cluster tampil')
print()
print('================================================================')
print('  CHECKLIST BETA TESTING (Pengguna / Mentor)')
print('================================================================')
print('  [ ] Tampilan responsif di berbagai ukuran layar')
print('  [ ] Loading data < 5 detik')
print('  [ ] Instruksi UI mudah dipahami pengguna non-teknis')
print('  [ ] Penyebab defect per batch mudah dibaca')
print('  [ ] Output prediksi model dapat diinterpretasi')
print('  [ ] Tidak ada crash saat filter berubah')
print('  [ ] Re-train model berjalan tanpa error')
print()
print('================================================================')
print('  STRUKTUR FILE')
print('================================================================')
print('  project/')
print('  |- app.py                          <- file utama dashboard')
print('  |- dataset_with_predictions.csv    <- output notebook 03')
print('  |- dataset_clean.csv               <- output notebook 01')
print('  |- 05_dashboard.ipynb              <- notebook ini')
print()
print('================================================================')
print('  NOTEBOOK 05 SELESAI - app.py siap dijalankan!')
print('================================================================')


  CHECKLIST ALPHA TESTING (Internal Tim)
  [ ] Dashboard terbuka di http://localhost:8501
  [ ] Upload CSV berhasil (dataset_with_predictions.csv)
  [ ] KPI cards tampil dengan angka benar
  [ ] Filter Bulan / Line / Status berfungsi
  [ ] Tab Overview: chart distribusi dan tabel data tampil
  [ ] Tab Monitoring: scatter yield + anomaly score tampil
  [ ] Tab Analisis Defect: reason cards muncul per batch
  [ ] Tab Model AI: tombol Re-train berhasil dijalankan
  [ ] Tab Clustering: PCA scatter dan profil cluster tampil

  CHECKLIST BETA TESTING (Pengguna / Mentor)
  [ ] Tampilan responsif di berbagai ukuran layar
  [ ] Loading data < 5 detik
  [ ] Instruksi UI mudah dipahami pengguna non-teknis
  [ ] Penyebab defect per batch mudah dibaca
  [ ] Output prediksi model dapat diinterpretasi
  [ ] Tidak ada crash saat filter berubah
  [ ] Re-train model berjalan tanpa error

  STRUKTUR FILE
  project/
  |- app.py                          <- file utama dashboard
  |- dataset_with_predictions

---
## Ringkasan Fitur Dashboard

| Komponen | Status |
|----------|--------|
| Upload Dataset (CSV) | Tersedia |
| KPI Cards (Total, Normal, Defect, Yield, Suhu) | Tersedia |
| Distribusi Defect (Pie, per Bulan, per Line) | Tersedia |
| Box Plot Parameter Kritis (Suhu & KA) | Tersedia |
| Tabel Batch Interaktif | Tersedia |
| Monitoring Yield per Batch + Moving Avg | Tersedia |
| Anomaly Score Isolation Forest | Tersedia |
| Scatter Suhu vs Yield | Tersedia |
| Analisis Penyebab Defect (Frekuensi + Cards) | Tersedia |
| Heatmap Korelasi Parameter Kritis | Tersedia |
| Re-train RF + GB + Isolation Forest | Tersedia |
| Metrik Evaluasi (Accuracy, Precision, Recall, F1) | Tersedia |
| Confusion Matrix (semua model) | Tersedia |
| Feature Importance Top 20 | Tersedia |
| Clustering K-Means + Elbow Method | Tersedia |
| PCA 2D Scatter (Cluster & Status) | Tersedia |
| Filter dinamis Bulan / Line / Status | Tersedia |

**Proyek PJK-GM016 | Pijak x IBM SkillsBuild**
